# 02 Split and evaluation

**Purpose.** The foundation every later notebook builds on, so that all models are compared on the same searches with the same metric:

1. a **search table**: one row per search with list size, labels and history context (used for the split and for segments);
2. one fixed **train / validation / test split** by whole search record;
3. the **evaluation** (in `jdsearch.py`), checked here;
4. the first reference result: **random order** on the validation set.

**Split.**

| Split | Records | Rule |
|---|---:|---|
| Test | 20,000 | Drawn only from records outside the 5,000 explored before the split; scored once, in notebook 07 |
| Validation | 20,000 | Also outside the explored records; used for all model comparisons |
| Train | 133,831 | Everything else, including the explored records |
| Dev subsample | 40,000 | Fixed subset of train for quick experiments |

Searches without an engaged candidate stay in their split but are excluded from evaluation (every ordering scores the same).

**Clamped final gap.** In 5.3% of records the gap between the last history action and the search is exactly 1 second, while 0 never occurs and 2 seconds is 84 times rarer. Most likely gaps of zero or less were clamped to 1 second, so these histories contain actions from during or after the search (label leakage). The search table flags them (`final_gap_clamped`): they stay in their split, are excluded from training, and the main evaluation population `validation_clean` leaves them out (all searches serve as a sensitivity check).

**Runtime:** CPU, about 5 minutes (three passes over the behaviour file).

In [1]:
# Setup: on Colab, clone the repository (shared code in jdsearch.py) and use Google Drive as the working folder.
import subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/repo").is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/liamkunert/BT4222-group17-jdsearch-submission.git", "/content/repo"], check=True)
    sys.path.insert(0, "/content/repo")
else:
    sys.path.insert(0, str(Path.cwd().resolve().parent))
import jdsearch as jd

paths = jd.setup()
jd.require(paths, "data/user_behavior_data.txt")

Mounted at /content/drive
working folder: /content/drive/MyDrive/BT4222-G17-JDsearch


In [2]:
import hashlib
import json

import numpy as np
import pandas as pd

SPLIT_SEED = 4222
TEST_SIZE, VALIDATION_SIZE, DEV_SIZE = 20_000, 20_000, 40_000
pd.options.display.float_format = "{:,.4f}".format

## 1. Search table

One streaming pass. Per search: number of candidates, engaged and purchased candidates, history length by action type, actions in the last 7 days, days since the last action and the final gap. Nothing here depends on candidate file order.

In [3]:
searches = jd.search_table(paths.behaviour)
searches.to_parquet(paths.processed / "search_table.parquet")
print(f"searches: {len(searches):,}")
searches.describe(percentiles=[0.5]).T

searches: 173,831


,count,mean,std,min,50%,max
n_candidates,"173,831.0000",89.2247,38.8574,1.0000,90.0000,200.0000
n_engaged,"173,831.0000",1.8460,1.7200,0.0000,1.0000,52.0000
n_purchased,"173,831.0000",0.1199,0.3373,0.0000,0.0000,12.0000
history_len,"173,831.0000",153.4091,224.8016,1.0000,78.0000,"4,802.0000"
n_click,"173,831.0000",47.7412,93.3323,0.0000,17.0000,"3,575.0000"
n_cart,"173,831.0000",66.3087,115.0793,0.0000,29.0000,"3,614.0000"
n_ord,"173,831.0000",39.2179,70.4069,0.0000,15.0000,"2,962.0000"
n_flw,"173,831.0000",0.1414,4.0069,0.0000,0.0000,695.0000
actions_last_7d,"173,831.0000",56.6034,106.3307,0.0000,21.0000,"3,850.0000"
days_since_last_action,"173,831.0000",5.4011,21.8621,0.0000,0.2232,363.9587


## 2. The explored sample

Before the split, 5,000 records were explored (`numpy.random.default_rng(4222).choice(n, 5000, replace=False)`; notebook 01 uses the same draw). They shaped our feature choices, so they must not be in validation or test.

In [4]:
explored = jd.explored_ids(len(searches))
print(f"explored records: {len(explored):,}")

explored records: 5,000


## 3. Assign the split

All records outside the explored sample are shuffled once with a fixed seed: the first 20,000 become test, the next 20,000 validation, the rest join the explored records in train. The dev subsample is a fixed random subset of train.

In [5]:
rng = np.random.default_rng(SPLIT_SEED)
eligible = rng.permutation(np.setdiff1d(searches.index.to_numpy(), explored))
split = pd.Series("train", index=searches.index, name="split")
split.loc[eligible[:TEST_SIZE]] = "test"
split.loc[eligible[TEST_SIZE:TEST_SIZE + VALIDATION_SIZE]] = "validation"
train_ids = split.index[split == "train"].to_numpy()
dev = pd.Series(False, index=searches.index, name="dev_subsample")
dev.loc[rng.choice(train_ids, DEV_SIZE, replace=False)] = True

assignment = pd.concat([split, dev], axis=1)
assert not assignment.loc[explored, "split"].isin(["validation", "test"]).any()
assert (assignment.dev_subsample & (assignment.split != "train")).sum() == 0
assignment.to_csv(paths.processed / "split_v1.csv")
meta = {"version": "split_v1", "seed": SPLIT_SEED, "records": len(assignment),
        "counts": assignment.split.value_counts().to_dict(), "dev_subsample": int(dev.sum()),
        "explored_seed": jd.EXPLORED_SEED, "explored_size": len(explored)}
(paths.processed / "split_v1.json").write_text(json.dumps(meta, indent=2))
assignment.split.value_counts().to_frame("records")

,records
split,
train,133831
validation,20000
test,20000


## 4. Balance check

A uniform split should give every split the same profile: list size, engagement, purchases, history length, recent activity and the clamped final gap.

In [6]:
table = searches.join(assignment)
table["recent_activity"] = table.days_since_last_action < 7


def profile(part):
    return pd.Series({
        "searches": len(part),
        "mean_candidates": part.n_candidates.mean(),
        "engaged_share_of_candidates": part.n_engaged.sum() / part.n_candidates.sum(),
        "share_searches_with_purchase": (part.n_purchased > 0).mean(),
        "share_searches_no_engagement": (part.n_engaged == 0).mean(),
        "median_history_len": part.history_len.median(),
        "share_activity_last_7d": part.recent_activity.mean(),
        "share_final_gap_clamped": part.final_gap_clamped.mean(),
    })


groups = {"train": table[table.split == "train"], "dev subsample": table[table.dev_subsample],
          "validation": table[table.split == "validation"], "test": table[table.split == "test"]}
pd.DataFrame({name: profile(part) for name, part in groups.items()})

,train,dev subsample,validation,test
searches,"133,831.0000","40,000.0000","20,000.0000","20,000.0000"
mean_candidates,89.2433,89.3341,89.0786,89.2456
engaged_share_of_candidates,0.0207,0.0207,0.0209,0.0207
share_searches_with_purchase,0.1171,0.1192,0.1169,0.1148
share_searches_no_engagement,0.0122,0.0126,0.0114,0.0119
median_history_len,78.0000,78.0000,78.0000,78.0000
share_activity_last_7d,0.8842,0.8821,0.8836,0.8832
share_final_gap_clamped,0.0529,0.0529,0.0532,0.0551


## 5. The evaluation

`jdsearch.evaluate` scores each search's complete candidate list:

- **NDCG@10** with linear gains 0/1/2/3 (primary metric), **Recall@10** (share of engaged candidates in the top 10) and **MRR** (1 / rank of the first engaged candidate);
- **ties:** the file order puts engaged candidates first, so it must never break ties. Each metric is the exact expectation over a random order within tied scores; a model that gives every candidate the same score gets exactly the random-order value;
- searches without an engaged candidate are excluded.

The checks below run hand-computed cases and compare every metric with a brute-force average over all orderings of small tied lists.

In [7]:
from itertools import permutations

jd.validate()
check_rng = np.random.default_rng(0)
for _ in range(200):
    n = check_rng.integers(2, 7)
    scores, labels = check_rng.integers(0, 2, n), check_rng.choice(4, n, p=[0.6, 0.2, 0.1, 0.1])
    if not (labels >= 1).any():
        continue
    # brute force: average the metrics over all orders that are consistent with the scores (ties in any order)
    orders = [p for p in permutations(range(n)) if all(scores[p[i]] >= scores[p[i + 1]] for i in range(n - 1))]
    brute = np.mean([jd.expected_metrics(np.arange(n, 0, -1)[np.argsort(p)], labels, k=3) for p in orders], axis=0)
    assert np.allclose(jd.expected_metrics(scores, labels, k=3), brute)
print("evaluation checks passed")

evaluation checks passed


## 6. Reference points on the validation set

The validation candidates (shuffled within each search) and two reference rankings:

- **Random order** (every candidate gets the same score): the floor every model must beat.
- **Oracle** (score = true label): the ceiling, 1.0 by construction; it only checks the pipeline.

In [8]:
validation_ids = assignment.index[assignment.split == "validation"].to_numpy()
validation = jd.load_candidates(paths.behaviour, validation_ids)
validation_sizes = searches.loc[validation_ids, "n_candidates"]
print(f"validation: {validation.session_id.nunique():,} searches, {len(validation):,} candidates")

validation["score_random"] = 0.0
validation["score_oracle"] = validation.label.astype(float)
per_random = jd.evaluate(validation, "score_random", expected_sizes=validation_sizes)
per_oracle = jd.evaluate(validation, "score_oracle", expected_sizes=validation_sizes)
pd.concat({"random order": jd.summarise(per_random).iloc[0], "oracle": jd.summarise(per_oracle).iloc[0]}, axis=1).T

validation: 20,000 searches, 1,781,573 candidates


,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr
random order,"19,772.0000",0.0845,0.0830,0.0860,0.1599,0.1020
oracle,"19,772.0000",1.0000,1.0000,1.0000,0.9984,1.0000


Random order by segment. The random value depends on how many candidates a search has and how many are engaged, so models are compared with random **within** a segment.

In [9]:
segments = table.loc[per_random.index]
recent = segments.recent_activity.map({True: "activity in last 7 days", False: "no activity in last 7 days"})
list_size = pd.cut(segments.n_candidates, [0, 30, 90, 150, 200], labels=["1-30", "31-90", "91-150", "151-200"])
pd.concat({"recent activity": jd.summarise(per_random, recent), "list size": jd.summarise(per_random, list_size)})

searches  ndcg_at_10  \
recent activity activity in last 7 days    17,462.0000      0.0849   
                no activity in last 7 days  2,310.0000      0.0816   
list size       1-30                        1,825.0000      0.3312   
                31-90                      10,153.0000      0.0668   
                91-150                      6,003.0000      0.0535   
                151-200                     1,791.0000      0.0371   

                                            ndcg_ci_low  ndcg_ci_high  \
recent activity activity in last 7 days          0.0833        0.0864   
                no activity in last 7 days       0.0779        0.0859   
list size       1-30                             0.3207        0.3417   
                31-90                            0.0663        0.0673   
                91-150                           0.0532        0.0538   
                151-200                          0.0366        0.0376   

                                            recall_at_10    mrr  
recent activity activity in last 7 days           0.1605 0.1023  
                no activity in last 7 days        0.1550 0.0998  
list size       1-30                              0.5874 0.2885  
                31-90                             0.1371 0.0815  
                91-150                            0.0992 0.0847  
                151-200                           0.0568 0.0860

## 7. Results table

Every evaluated ranking of the pipeline is written to `results.csv` in the working folder with `jd.log_result`; notebook 07 reads it back. Random order on all validation searches and on `validation_clean`:

In [10]:
clean_ids = per_random.index[~searches.final_gap_clamped.reindex(per_random.index).to_numpy()]
for split_name, ids in [("validation", per_random.index), ("validation_clean", clean_ids)]:
    jd.log_result(paths, "02", "references", "random order", split_name, per_random.loc[ids],
                  notes="all candidates tied; exact expectation over random order")
jd.read_results(paths)

,date,notebook,step,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
0,2026-10-30,2,references,random order,validation,19772,0.0845,0.0830,0.0860,0.1599,0.1020,6aeeebd,all candidates tied; exact expectation over ra...
1,2026-10-30,2,references,random order,validation_clean,18844,0.0842,0.0828,0.0858,0.1587,0.1026,6aeeebd,all candidates tied; exact expectation over ra...


## 8. Guardrails

Four checks that protect every later comparison:

1. **Exact duplicate records** in the behaviour file (a duplicate across splits would leak).
2. **Completeness:** `evaluate(..., expected_sizes=...)` fails if a search is missing or a list is incomplete; `paired_difference` fails if two results cover different searches.
3. **Candidate order:** `load_candidates` shuffles candidates within each search (seeded). In the raw file the first candidate is almost always engaged; after shuffling it should be engaged about as often as any other.
4. **Permutation invariance:** shuffling the rows of a scored frame must not change any metric.

In [11]:
hashes = pd.Series([hashlib.md5("\t".join(fields).encode()).hexdigest() for _, fields in jd.iter_records(paths.behaviour)])
duplicates = hashes.duplicated(keep=False)
print(f"records in exact duplicate groups: {duplicates.sum():,}")
if duplicates.any():
    print(pd.Series(assignment.split.to_numpy()[duplicates.to_numpy()]).value_counts().to_string())

records in exact duplicate groups: 0


In [12]:
# 2. Completeness: a truncated list and a missing search must both be rejected.
for name, broken in {"truncated list": validation.drop(validation.index[0]),
                     "missing search": validation[validation.session_id != validation_ids[0]]}.items():
    try:
        jd.evaluate(broken, "score_random", expected_sizes=validation_sizes)
    except ValueError as error:
        print(f"{name}: rejected ({error})")
    else:
        raise AssertionError(f"{name} was accepted")
try:
    jd.paired_difference(per_random, per_random.iloc[1:])
except ValueError as error:
    print(f"paired comparison on different searches: rejected ({error})")
else:
    raise AssertionError("paired comparison on different searches was accepted")

# 3. Candidate order: engaged share of the first candidate per search, raw file order vs shuffled.
raw_order = jd.load_candidates(paths.behaviour, validation_ids[:2000], shuffle_seed=None)
first_raw = raw_order.groupby("session_id").label.first().ge(1).mean()
first_shuffled = validation.groupby("session_id").label.first().ge(1).mean()
print(f"first candidate engaged: raw file order {first_raw:.1%}, shuffled {first_shuffled:.1%} "
      f"(overall engaged share {validation.label.ge(1).mean():.1%})")
assert first_shuffled < 0.1

# 4. Permutation invariance with a tie-heavy score.
validation["score_check"] = np.random.default_rng(1).integers(0, 3, len(validation)).astype(float)
before = jd.evaluate(validation, "score_check")
after = jd.evaluate(validation.sample(frac=1, random_state=2), "score_check")
assert np.allclose(before.to_numpy(), after.loc[before.index].to_numpy())
print("metrics unchanged after shuffling rows")

truncated list: rejected (1 searches have incomplete or extra candidates)
missing search: rejected (1 expected searches missing, 0 unexpected searches present)
paired comparison on different searches: rejected (Paired comparison needs exactly the same searches in both results)
first candidate engaged: raw file order 98.9%, shuffled 3.3% (overall engaged share 2.1%)
metrics unchanged after shuffling rows


## Summary

- **Outputs** (working folder, `processed/`): `search_table.parquet`, `split_v1.csv` and `split_v1.json`; regenerated identically by this notebook.
- **Splits** have the same profile (section 4); the explored records are only in train.
- **Evaluation:** exact expected NDCG@10, Recall@10 and MRR under random tie-breaking, checked against hand cases and brute force.
- **Floor to beat:** random order, NDCG@10 in section 6 (logged in `results.csv`).
- **Guardrails:** no duplicate leak, complete lists enforced, candidate order shuffled, metrics invariant to row order.